In [ ]:

from types import SimpleNamespace
import numpy as np
import math

class AM81111:

    encoderUnit = SimpleNamespace(**{ 
        'bits': [16, 16]
    })

    driveUnit = SimpleNamespace(**{ 
        'gearBoxGearRatio': 1000.0,        
        'timingBeltTransmissionGearRatio': 2.0,
        'spindlePitch': 5.0,
        'motorIncrementPositions': 262_144, 
        'cylinderDiameter': 15.0,
        'limit': {
            'low': 0,
            'high': 24_185_993 
        }
    })

    cylinderUnit = SimpleNamespace(**{ 
        'limit': {
            'low':   81.3,  # mm bottom
            'high': 131.1   # mm top
        }
    })

    def gearRatio(self):
        return self.driveUnit.timingBeltTransmissionGearRatio * self.driveUnit.gearBoxGearRatio

    def cylinderArea(self):
        return np.pow(self.driveUnit.cylinderDiameter,2) * np.pi / 4.
    
    def pitchVolume(self):
        return self.driveUnit.spindlePitch * self.cylinderArea()

    def mulmin2incs(self, value):
        # mm/r
        transmission = self.driveUnit.spindlePitch / self.gearRatio()
        # µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        # µl/min
        return np.clip(value / (injectionRateIncrement * 60), self.driveUnit.limit['low'], self.driveUnit.limit['high'])
    
    def incs2mulmin(self, value):
        # mm/r
        transmission = self.driveUnit.spindlePitch / self.gearRatio()
        # mm³/r ~ µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        return (value * injectionRateIncrement * 60.0)
    
    def mms2mulmin(self, value):
        # mm³/s ~ µl/s
        return value * self.cylinderArea() * 60
    
    def turn2ml(self, value):

        bitRange = 2 ** self.encoderUnit.bits[1] -1
        
        mtb = value[0] * self.pitchVolume() / self.gearRatio()
        stb = value[1] / bitRange * self.pitchVolume() / self.gearRatio()
        
        return (mtb + stb) / 1000

    def ml2turn(self, value):
        bitRange = 2 ** self.encoderUnit.bits[1] -1
        mtb = round(np.trunc(value / self.pitchVolume() * self.gearRatio() * 1000), 0)
        stb = round(bitRange * (value - self.turn2ml([mtb, 0])) / self.pitchVolume() * self.gearRatio() * 1000, 0)
        return [np.uint32(mtb), np.uint32(stb)]
    
    def value(self, value, range=32):
        rc = (2**range - 1) + value if value < 0 else value
        return rc
    
    def split(self, value, bits, range=32):            
        value = bin(value)[2:].zfill(range)
        return [
            int(value[:bits].zfill(range),2), 
            int(value[bits:].zfill(range),2)
            ]
    
    def merge(self, value, bits, range=32, verbose=False):          
        rc = self.value(int("".join([
            bin(value[0])[2:].zfill(bits), 
            bin(value[1])[2:].zfill(range-bits)]), 2), range)
        return rc

INCS_MAX = 24_185_993

am = AM81111()
am.incs2mulmin(INCS_MAX)


np.float64(2445.612578477266)

In [ ]:
A = np.pow(15,2) * np.pi / 4.

# time in seconds for 10% of the operational range

4.98 / (np.arange(100, 2000, 100) / ( 60 * A))

array([519.54088509, 259.77044254, 173.18029503, 129.88522127,
       103.90817702,  86.59014751,  74.22012644,  64.94261064,
        57.72676501,  51.95408851,  47.23098955,  43.29507376,
        39.96468347,  37.11006322,  34.63605901,  32.47130532,
        30.56122853,  28.8633825 ,  27.34425711])

In [ ]:
A = np.pow(15,2) * np.pi / 4.   # mm²
sp = 5.0                        # mm

mt = 1500 / 2000

mul = sp * A * mt               

mul / 1000.0

np.float64(0.662679700366597)

In [7]:
class AM81111Profile:

    # status word
    NOT_READY_TO_SWITCH_ON  = 0     # xxx0 xxxx x0xx 0000
    READY_TO_SWITCH_ON      = 1     # xxx0 xxxx x01x 0001
    SWITCHED_ON             = 2     # xxx0 xxxx x1xx 0011
    OPERATION_ENABLED       = 4     # xxx0 xxxx x1xx 0111
    FAULT                   = 8     # xxx0 xxxx x0xx 1000
    
    QUICK_STOP              = 32
    SWITCH_ON_DISABLED      = 64    # xxx0 xxxx x1xx 0000
    WARNING                 = 128
    LIMIT_ACTIVE            = 2_048

    # control word LoByte
    FAULT_RESET             = '10000000'    # 1xxx xxxx 15
    SHUTDOWN                = '00000110'    # 0xxx x110 2,6,8
    SWITCH_ON               = '00000111'    # 0xxx 0111 3,5
    ENABLE_OPERATION        = '00001111'    # 0xx0 1111 4

    control = ['10000000','00000110','00000111','00001111']
    control_name = ['FAULT_RESET','SHUTDOWN','SWITCH_ON','ENABLE_OPERATION']

    @staticmethod
    def __control__(value):
        for i,c in enumerate(AM81111Profile.control):
            if int(value,2) == int(c,2):
                return value, AM81111Profile.control_name[i]
        return value, 'UNKNOWN'
    
    """
    status                                              control

    xxx0 xxxx x0xx 0000     not ready to switch on
               |                                        
    xxx0 xxxx x1xx 0000     switch on disabled
                      |                                 0000 0000 0xxx x110       shutdown
    xxx0 xxxx x01x 0001     ready to switch on
                     |                                  0000 0000 0xxx x111       switch on
    xxx0 xxxx x1xx 0011     switched on
                    |                                   0000 0000 0xx0 1111       enable operation
    xxx0 xxxx x1xx 0111     operation enabled


    xxx0 xxxx x0xx 1111     fault reaction active
    xxx0 xxxx x0xx 1000     fault                       0000 0000 1xxx xxxx       fault reset

    
                                                        0000 0000 0xxx xx0x       disable voltage
                                                        0000 0000 0xxx x01x       
    """
    
    status = [
        NOT_READY_TO_SWITCH_ON, READY_TO_SWITCH_ON, SWITCHED_ON, OPERATION_ENABLED, 
        FAULT, QUICK_STOP, SWITCH_ON_DISABLED, WARNING, LIMIT_ACTIVE]
    
    status_name = [
        'NOT_READ_TO_SWITCH_ON','READY_TO_SWITCH_ON', 'SWITCHED_ON', 'OPERATION_ENABLED', 
        'FAULT', 'QUICK_STOP', 'SWITCH_ON_DISABLED', 'WARNING', 'LIMIT_ACTIVE']
    
    @staticmethod
    def __status__(state):
        return ",".join([f"{AM81111Profile.status_name[i]}" 
                        for i,s in enumerate(AM81111Profile.status) 
                            if ((s & state) == s) and (s not in [0])
                        ])
    @staticmethod
    def __get__(state):
        return [s for s in AM81111Profile.status if ((s & state) == s) and (s not in [0])]

    @staticmethod
    def __has__(value, state):
        return (state | value) == value

    @staticmethod
    def __transit__(value):

        # transition, state, index

        if AM81111Profile.__has__(value, AM81111Profile.FAULT):
            return [
                (AM81111Profile.FAULT_RESET, AM81111Profile.SWITCH_ON_DISABLED, 15)
                ]

        if AM81111Profile.__has__(value, AM81111Profile.OPERATION_ENABLED):
            return [
                (AM81111Profile.SWITCH_ON, AM81111Profile.SWITCHED_ON, 5), 
                (AM81111Profile.SHUTDOWN, AM81111Profile.READY_TO_SWITCH_ON, 8)
                ]
        
        if AM81111Profile.__has__(value, AM81111Profile.SWITCHED_ON):
            return [                
                (AM81111Profile.ENABLE_OPERATION, AM81111Profile.OPERATION_ENABLED, 4),
                (AM81111Profile.SHUTDOWN, AM81111Profile.READY_TO_SWITCH_ON, 6)
                ]
        
        if AM81111Profile.__has__(value, AM81111Profile.SWITCHED_ON):
            return [
                (AM81111Profile.ENABLE_OPERATION, AM81111Profile.OPERATION_ENABLED, 4)
                ]   
        
        if AM81111Profile.__has__(value, AM81111Profile.READY_TO_SWITCH_ON):
            return [
                (AM81111Profile.SWITCH_ON, AM81111Profile.SWITCHED_ON, 3),                
                ]

        if AM81111Profile.__has__(value, AM81111Profile.SWITCH_ON_DISABLED):
            return [
                (AM81111Profile.SHUTDOWN, AM81111Profile.READY_TO_SWITCH_ON, 2)
                ]    

        return []    
    
    warning_name = [None, None, 'UNDER_VOLTAGE', 'OVER_VOLTAGE', 'OVER_TEMPERATURE', 
                    'I2T_AMPLIFIER', 'I2T_MOTOR', 'ENCODER']
    
    error_name = ['ADC_ERROR', 'OVER_CURRENT', 'UNDER_VOLTAGE', 'OVER_VOLTAGE', 'OVER_TEMPERATURE', 
                  'I2T_AMPLIFIER', 'I2T_MOTOR', 'ENCODER', 'WATCHDOG']
    
    touch_name = ['TP1_ENABLE', 'TP1_POS', 'TP1_NEG', None, None, None, None, 'TP1_INPUT',
                  'TP2_ENABLE', 'TP2_POS', 'TP2_NEG', None, None, None, None, 'TP2_INPUT']

    @staticmethod
    def __info__(value, mode='e'):
        value = bin(value)[2:].zfill(16)[::-1]
        value = list(map(lambda x: int(x), value))
        if mode == 'w':            
            return ",".join([f"{AM81111Profile.warning_name[i]}" 
                        for i,n in enumerate(AM81111Profile.warning_name) 
                            if n is not None and value[i] == 1
                        ])
        if mode == 'e':            
            return ",".join([f"{AM81111Profile.error_name[i]}" 
                        for i,n in enumerate(AM81111Profile.error_name) 
                            if n is not None and value[i] == 1
                        ])
        if mode == 't':
            return ",".join([f"{AM81111Profile.touch_name[i]}" 
                        for i,n in enumerate(AM81111Profile.touch_name) 
                            if n is not None and value[i] == 1
                        ])
        return ""
    
status = '0010010001101000'

AM81111Profile.__has__(int(status,2), AM81111Profile.SWITCHED_ON)

False